### Analyzing and Making model for Portugal's bank real dataset

In [2]:
import pandas as pd
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

URL = ('https://archive.ics.uci.edu/static/public/222/bank+marketing.zip')

outer = ZipFile(BytesIO(urlopen(URL).read()))
inner = ZipFile(BytesIO(outer.read('bank-additional.zip')))
df = pd.read_csv(
    inner.open('bank-additional/bank-additional.csv'), sep=';'
)

df.to_csv('bank.csv', sep=';', index=False)
print(df.shape)  # (4119, 21)

(4119, 21)


### analyzing data

In [5]:
# analyzing dataset
print(df.head())
print(df.describe())
print(df.info())

# check for mising and duplicates
# print(df.isnull().sum())
# print(df.duplicated().sum())
# no missing and no duplicates

# analyzing target
print(df['y'].value_counts()) # no = 3668 , yes = 451
print(df['y'].value_counts(normalize=True)*100) # no = 89.05074 , yes = 10.94926

   age          job  marital          education default  housing     loan  \
0   30  blue-collar  married           basic.9y      no      yes       no   
1   39     services   single        high.school      no       no       no   
2   25     services  married        high.school      no      yes       no   
3   38     services  married           basic.9y      no  unknown  unknown   
4   47       admin.  married  university.degree      no      yes       no   

     contact month day_of_week  ...  campaign  pdays  previous     poutcome  \
0   cellular   may         fri  ...         2    999         0  nonexistent   
1  telephone   may         fri  ...         4    999         0  nonexistent   
2  telephone   jun         wed  ...         1    999         0  nonexistent   
3  telephone   jun         fri  ...         3    999         0  nonexistent   
4   cellular   nov         mon  ...         1    999         0  nonexistent   

  emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.e

## baseline model

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


X = df.drop(columns=['y'])
y = df['y'].map({'no': 0, 'yes': 1}) # encode target

X = pd.get_dummies(X, drop_first=True) # one-hot encode other features

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# baseline model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9029126213592233
[[710  22]
 [ 58  34]]
              precision    recall  f1-score   support

           0       0.92      0.97      0.95       732
           1       0.61      0.37      0.46        92

    accuracy                           0.90       824
   macro avg       0.77      0.67      0.70       824
weighted avg       0.89      0.90      0.89       824



## feature engineering

In [22]:
# remove noisy/leakage features
# its better to do with new variable so we dont loose original dataset
df_clean = df.drop(columns=['duration', 'campaign'])

print(df_clean.shape) # (4119, 19)

X = df_clean.drop(columns=['y'])
y = df_clean['y'].map({'no': 0, 'yes': 1})

X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

(4119, 19)


## Bagging Classifier

In [30]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report

model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42
)
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(confusion_matrix(y_test, model.predict(X_test)))
print(classification_report(y_test, model.predict(X_test)))

# threshold ozgartirib korsak
y_proba = model.predict_proba(X_test)[:, 1]
y_pred_30 = (y_proba >= 0.3).astype(int)
print('====== Fixed Threshold ======')
print(accuracy_score(y_test, y_pred_30))
print(confusion_matrix(y_test, y_pred_30))
print(classification_report(y_test, y_pred_30))

0.8919902912621359
[[719  15]
 [ 74  16]]
              precision    recall  f1-score   support

           0       0.91      0.98      0.94       734
           1       0.52      0.18      0.26        90

    accuracy                           0.89       824
   macro avg       0.71      0.58      0.60       824
weighted avg       0.86      0.89      0.87       824

====== Fixed Threshold ======
0.866504854368932
[[675  59]
 [ 51  39]]
              precision    recall  f1-score   support

           0       0.93      0.92      0.92       734
           1       0.40      0.43      0.41        90

    accuracy                           0.87       824
   macro avg       0.66      0.68      0.67       824
weighted avg       0.87      0.87      0.87       824



In [31]:
from sklearn.metrics import precision_score, recall_score, f1_score
## experiment
for threshold in [0.2, 0.25, 0.3, 0.4, 0.5]:
    y_pred_temp = (y_proba >= threshold).astype(int)

    print("\nThreshold:", threshold)
    print("Precision:", precision_score(y_test, y_pred_temp))
    print("Recall:", recall_score(y_test, y_pred_temp))
    print("F1:", f1_score(y_test, y_pred_temp))


Threshold: 0.2
Precision: 0.30864197530864196
Recall: 0.5555555555555556
F1: 0.3968253968253968

Threshold: 0.25
Precision: 0.3412698412698413
Recall: 0.4777777777777778
F1: 0.39814814814814814

Threshold: 0.3
Precision: 0.3979591836734694
Recall: 0.43333333333333335
F1: 0.4148936170212766

Threshold: 0.4
Precision: 0.45901639344262296
Recall: 0.3111111111111111
F1: 0.3708609271523179

Threshold: 0.5
Precision: 0.5151515151515151
Recall: 0.18888888888888888
F1: 0.2764227642276423


## random forest

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print(accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

0.8932038834951457
[[721  13]
 [ 75  15]]
              precision    recall  f1-score   support

           0       0.91      0.98      0.94       734
           1       0.54      0.17      0.25        90

    accuracy                           0.89       824
   macro avg       0.72      0.57      0.60       824
weighted avg       0.87      0.89      0.87       824



## pruned random forest

In [25]:
rf_pruned = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

rf_pruned.fit(X_train, y_train)

y_pred_pruned = rf_pruned.predict(X_test)

print(accuracy_score(y_test, y_pred_pruned))
print(confusion_matrix(y_test, y_pred_pruned))
print(classification_report(y_test, y_pred_pruned)) # 0.16 vashe yomon for finding potential customers

0.8968446601941747
[[731   3]
 [ 82   8]]
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       734
           1       0.73      0.09      0.16        90

    accuracy                           0.90       824
   macro avg       0.81      0.54      0.55       824
weighted avg       0.88      0.90      0.86       824



## Ada Boosting

In [26]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

boost = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)
boost.fit(X_train, y_train)

print(accuracy_score(y_test, boost.predict(X_test)))
print(confusion_matrix(y_test, boost.predict(X_test)))
print(classification_report(y_test, boost.predict(X_test)))

0.9004854368932039
[[728   6]
 [ 76  14]]
              precision    recall  f1-score   support

           0       0.91      0.99      0.95       734
           1       0.70      0.16      0.25        90

    accuracy                           0.90       824
   macro avg       0.80      0.57      0.60       824
weighted avg       0.88      0.90      0.87       824



In [27]:
# check feature importance
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print(imp.head(10))

age                 0.159819
euribor3m           0.145065
nr.employed         0.060353
pdays               0.038309
cons.conf.idx       0.035553
housing_yes         0.035045
cons.price.idx      0.030489
emp.var.rate        0.029760
poutcome_success    0.025102
previous            0.023645
dtype: float64


### improving existing models

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

# check again with existing data our model
X_train_new, X_val, y_train_new, y_val = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42
)

model.fit(X_train_new, y_train_new)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeClassifier()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary <warm_start>`... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [34]:
y_proba = model.predict_proba(X_val)[:, 1]

for threshold in [0.2, 0.25, 0.3, 0.4, 0.5]:
    y_pred = (y_proba >= threshold).astype(int)

    print(
        threshold,
        'Precision:', round(precision_score(y_val, y_pred), 3),
        'Recall:', round(recall_score(y_val, y_pred), 3),
        'F1:', round(f1_score(y_val, y_pred), 3)
    )


# 0.3 the best one , with highest f-1 score
y_proba_test = model.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= 0.3).astype(int)

print('Accuracy:', accuracy_score(y_test, y_pred_test))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_test))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_test))

0.2 Precision: 0.304 Recall: 0.578 F1: 0.398
0.25 Precision: 0.361 Recall: 0.533 F1: 0.43
0.3 Precision: 0.434 Recall: 0.478 F1: 0.455
0.4 Precision: 0.5 Recall: 0.389 F1: 0.438
0.5 Precision: 0.568 Recall: 0.233 F1: 0.331
Accuracy: 0.8640776699029126

Confusion Matrix:
[[675  59]
 [ 53  37]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.92      0.92       734
           1       0.39      0.41      0.40        90

    accuracy                           0.86       824
   macro avg       0.66      0.67      0.66       824
weighted avg       0.87      0.86      0.87       824

